In [18]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [19]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍


In [10]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/simpleserial-xor
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
Compiling:
.
-en     simpleserial-xor.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
Compiling:
-en     .././hal//sam4s/startup_sam4s.c ...
.
.
Compiling:
.
Compiling:
.
-en     .././hal//sam4s/sam4s_hal.c ...
Compiling:
-en     .././hal//sam4s/uart.c ...
Compiling:
-en     .././hal//sam4s/pio.c ...
-en     .././hal//sam4s/system_sam4s.c ...
.
Compiling:
.
-en     .././hal//sam4s/sysclk.c ...
Compiling:
-en     .././hal//sam4s/pmc.c ...
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     simp

In [12]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-xor/simpleserial-xor-{}.hex".format(PLATFORM))

In [20]:

print(target)

SimpleSerial2 Settings =
	output_len     = 16
	baud           = 230400
	xonxoff        = None
	currently_xoff = None
	parity         = none
	stop_bits      = 1
	


In [25]:
target.flush()

target.simpleserial_write('w', bytearray())

cmds = target.simpleserial_read('r', 32, ack=False)

print("Raw:", cmds)

if cmds:
    print("Commands:", [chr(x) for x in cmds])

(ChipWhisperer Target WARNING|File SimpleSerial2.py:585) Packet indicates different length than received 4 vs 32


Raw: CWbytearray(b'76 77 61 62')
Commands: ['v', 'w', 'a', 'b']


In [29]:
# One useful correction to the w test
target.flush()

target.simpleserial_write('w', bytearray())

cmds = target.simpleserial_read('r', 4, ack=False)

print("Raw:", cmds)
print("Commands:", [chr(x) for x in cmds])

Raw: CWbytearray(b'76 77 61 62')
Commands: ['v', 'w', 'a', 'b']


In [26]:
# The next issue is probably the V2 ACK/status packet sequencing, rather than an unknown command.

ktp = cw.ktp.Basic()
key, text = ktp.next()

print("key :", key)
print("text:", text)

target.flush()

# Set key
target.simpleserial_write('b', key)

# Explicitly consume the V2 acknowledgement
ack = target.simpleserial_wait_ack()

print("b ACK:", ack)

key : CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
text: CWbytearray(b'1d 87 55 63 ac c8 78 81 b1 ed 66 3c ac b8 95 1b')
b ACK: bytearray(b'\x00')


In [27]:
# Then test a separately

target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

print("response:", response)

expected = bytearray(t ^ k for t, k in zip(text, key))

print("received:", response)
print("expected:", expected)
print("match   :", response == expected)

response: CWbytearray(b'36 f9 40 75 84 66 aa 27 1a 1a 73 b4 a5 77 da 27')
received: CWbytearray(b'36 f9 40 75 84 66 aa 27 1a 1a 73 b4 a5 77 da 27')
expected: bytearray(b"6\xf9@u\x84f\xaa\'\x1a\x1as\xb4\xa5w\xda\'")
match   : True


In [76]:
# Then test only b:
ktp = cw.ktp.Basic()
key, text = ktp.next()

print("key:", key)

target.simpleserial_write('b', key)

ack = target.simpleserial_wait_ack()
print("b ack:", ack)

print("remaining:", target.in_waiting())

key: CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
b ack: bytearray(b'\x00')
remaining: 0


In [79]:
# test a:
target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

print("response:", response)

expected = bytearray(t ^ k for t, k in zip(text, key))

print("expected:", expected)
print("match:", response == expected)

response: CWbytearray(b'10 25 44 02 45 ad 42 81 a7 5f ae 95 ca 3a e2 99')
expected: bytearray(b'\x10%D\x02E\xadB\x81\xa7_\xae\x95\xca:\xe2\x99')
match: True


In [95]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

# --------------------------------
# Set key
# --------------------------------

target.simpleserial_write('b', key)

ack = target.simpleserial_wait_ack()

if ack != bytearray([0]):
    raise RuntimeError(f"Set-key failed: {ack}")

# --------------------------------
# XOR
# --------------------------------

target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

# --------------------------------
# Verify
# --------------------------------

expected = bytearray(t ^ k for t, k in zip(text, key))

print("key     :", key)
print("text    :", text)
print("response:", response)
print("expected:", expected)

if response is not None:
    print("match   :", bytes(response) == bytes(expected))

key     : CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
text    : CWbytearray(b'4b b4 e9 e1 da b1 ac 3c b0 1e f6 23 2b c3 a9 f9')
response: CWbytearray(b'60 ca fc f7 f2 1f 7e 9a 1b e9 e3 ab 22 0c e6 c5')
expected: bytearray(b'`\xca\xfc\xf7\xf2\x1f~\x9a\x1b\xe9\xe3\xab"\x0c\xe6\xc5')
match   : True


In [93]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

target.simpleserial_write('b', key)
target.simpleserial_write('a', text)

print("waiting:", target.in_waiting())
print("read:", repr(target.read(timeout=1000)))
#response = target.simpleserial_read('r', 16)
#print(response)

#expected = bytearray(t ^ k for t, k in zip(text, key))

#print("received:", response)
#print("expected:", expected)
#print("match   :", response == expected)

waiting: 0
read: ''
